# Stage 3 — pilot inference on Colab (T4)
Runs the 3 models on the 60 pilot samples with both prompt families (`native`, `json`) and saves the raw
outputs to Google Drive, so nothing is lost if Colab disconnects (re-running a cell skips finished samples).

Input: this repository (cloned from GitHub) + `pilot_data.zip` on Google Drive (made locally with
`python src/pack_pilot_data.py`). Output: `results/raw/*.jsonl` in the Drive folder below.

Runtime → Change runtime type → **T4 GPU**.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Settings (edit these)

In [ ]:
REPO_URL = "https://github.com/nilucmak/Structured-Report-Generation-from-Aerial-and-Satellite-Images-with-VLMs.git"
DRIVE_DIR = "/content/drive/MyDrive/TEZ"        # must contain pilot_data.zip; raw results are written here

## 2. Clone the code, install the pinned packages, mount Drive

In [ ]:
import os
if not os.path.exists("/content/repo"):
    !git clone -q {REPO_URL} /content/repo
%cd /content/repo
!git log -1 --oneline
!pip install -q -r requirements.txt

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
if not os.path.exists("/content/pilot_data"):
    !unzip -q {DRIVE_DIR}/pilot_data.zip -d /content/pilot_data

## 3. Point the config at the Colab folders
`configs/config.yaml` stays the single settings file; only the two paths are changed in this Colab copy.

In [ ]:
import yaml
cfg_path = "configs/config.yaml"
cfg = yaml.safe_load(open(cfg_path))
cfg["paths"]["data_root"] = "/content/pilot_data"
cfg["paths"]["raw_dir"] = f"{DRIVE_DIR}/results/raw"
yaml.safe_dump(cfg, open(cfg_path, "w"), sort_keys=False)
print(cfg["paths"])

## 4. Checks without the GPU: tests + dry run (all 60 images found, prompts printed)

In [ ]:
!python tests/parser_test.py | tail -1
!python tests/schema_test.py | tail -1
!python src/run_inference.py --model qwen2vl_7b --family native --dry-run | tail -1

## 5. First a tiny run: 1 sample per dataset for each model
Check the printed seconds / VRAM and look at the raw outputs (cell 7) before the full run.
Watch for empty or `!!!!` outputs: that would be a float16 overflow (T4 has no bfloat16).

In [ ]:
for model in ["qwen2vl_7b", "qwen25vl_3b", "geovlm_r1"]:
    !python src/run_inference.py --model {model} --family native --limit 1

## 6. Full pilot: 3 models × 2 prompt families × 60 samples

In [ ]:
for model in ["qwen2vl_7b", "qwen25vl_3b", "geovlm_r1"]:
    for family in ["native", "json"]:
        !python src/run_inference.py --model {model} --family {family}

## 7. Quick look: lines, parse success, truncation, time and VRAM per file

In [ ]:
import glob, json, statistics, sys
sys.path.insert(0, "src")
from box_parsers import parse_record
for path in sorted(glob.glob(f"{DRIVE_DIR}/results/raw/*.jsonl")):
    recs = [json.loads(l) for l in open(path) if l.strip()]
    parsed = [parse_record(r) for r in recs]
    secs = [r["seconds"] for r in recs]
    vram = [r["peak_vram_gb"] for r in recs if r["peak_vram_gb"] is not None]
    print(f"{os.path.basename(path)}: n={len(recs)}, parsed_ok={sum(p['ok'] for p in parsed)}, "
          f"with_boxes={sum(bool(p['boxes']) for p in parsed)}, truncated={sum(r['truncated'] for r in recs)}, "
          f"time={statistics.mean(secs):.1f}±{statistics.pstdev(secs):.1f} s, peak VRAM max={max(vram, default=0):.2f} GB")

In [ ]:
# One raw output per model and task, to see the real formats
seen = set()
for path in sorted(glob.glob(f"{DRIVE_DIR}/results/raw/*__native__*.jsonl")):
    for l in open(path):
        r = json.loads(l)
        if (r["model"], r["task"]) not in seen:
            seen.add((r["model"], r["task"]))
            print(f"=== {r['model']} / {r['task']} / {r['sample_id']}\n{r['raw_output'][:600]}\n")